# 滑动窗口稀疏注意力（Sliding Window Attention）

源码导航：[core/attention/sliding_window.py](../../../core/attention/sliding_window.py) 中的 `SlidingWindowAttention`、`make_sliding_window_mask`。

Mistral 7B 与 Longformer 采用的局部窗口注意力：位置 $i$ 只能 attend 到 $[\max(0, i-W+1), i]$ 内的 key。复杂度从 $O(T^2)$ 降至 $O(T \cdot W)$。

### 1. 掩码构造

因果 + 窗口约束：$M_{ij} = 1$ 当且仅当 $j \le i$ 且 $i - j < W$。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.attention.sliding_window import SlidingWindowAttention, make_sliding_window_mask

### 2. 注意力模式可视化

In [ ]:
T, W = 16, 4
mask = make_sliding_window_mask(T, W, torch.device('cpu')).float()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
full = torch.tril(torch.ones(T, T))
axes[0].imshow(full, cmap='Blues'); axes[0].set_title('Full Causal')
axes[1].imshow(mask, cmap='Blues'); axes[1].set_title(f'Window W={W}')
plt.tight_layout(); plt.show()

### 3. Forward 形状检查

In [ ]:
torch.manual_seed(0)
B, T, C = 2, 16, 64
x = torch.randn(B, T, C)

attn = SlidingWindowAttention(
    n_embd=C, n_head=4, n_head_kv=2, window_size=4, attn_impl='eager'
)
y = attn(x)
assert y.shape == (B, T, C)
print('output:', tuple(y.shape), 'window:', attn.window_size)

---

## 延伸阅读

- Mistral 7B 技术报告（滑动窗口 + GQA）
- Beltagy et al., *Longformer* (2020). [arXiv:2004.05150](https://arxiv.org/abs/2004.05150)